# Person 3 primary AKI baseline models

**COMP90089 Group 7 · Updated 16 October 2026 · Version 2**

This notebook develops four comparators for the primary AKI outcome: training
prevalence, age/gender logistic regression, structured L2 logistic regression,
and structured-plus-chief-complaint L2 logistic regression. It reuses Person 2's
frozen patient split and fitted lr_structured/lr_text preprocessing.
The code cells contain the complete Person 3 workflow; `aki_preprocessing.py` is
the byte-identical module supplied by Person 2.

Only training and validation data are loaded. Stage >=2, sensitivity
analyses and final test evaluation are outside this stage. AP is the provisional
development selection metric; the shared final evaluation and threshold protocol
still needs agreement with Person 5. Validation results are development results.


## Environment and file locations

Install the pinned packages in `requirements.txt` into a dedicated environment
and restart the notebook kernel if needed. The core scientific package versions
match Person 2's saved preprocessing. Python 3.12 was used to execute this notebook.

Local example, from `ML For Health`:

```sh
python3 -m venv .venv-person3
.venv-person3/bin/python -m pip install -r "Group Project/person3_handoff/requirements.txt"
```

For Colab, upload/unzip the private team files first. Keep this layout:

```text
Group Project/
  aki_handoff/predictor_columns.csv
  person2_handoff/person2_split_preprocessing/
    aki_preprocessing.py
    outputs/...
  person3_handoff/
    aki_preprocessing.py
    P3_baseline_models.ipynb
```

Set `PROJECT_DIR` below if the project folder is elsewhere. In Colab, install
packages before importing them, for example:

```python
%pip install -r "/content/Group Project/person3_handoff/requirements.txt"
```

No BigQuery queries or credentials are needed for these baselines.


In [ ]:
import sys
from pathlib import Path

# Set to a folder containing person2_handoff, or keep None for automatic discovery.
PROJECT_DIR = None

search_bases = ([Path(PROJECT_DIR).expanduser().resolve()] if PROJECT_DIR is not None
                else [Path.cwd().resolve(), *Path.cwd().resolve().parents])
module_dirs = []
for base in search_bases:
    module_dirs.extend([base, base / "person3_handoff",
                        base / "Group Project/person3_handoff"])
module_dir = next((p for p in module_dirs if (p / "aki_preprocessing.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError("Keep aki_preprocessing.py beside this notebook, or set PROJECT_DIR.")
sys.path.insert(0, str(module_dir))


## Predefined modelling choices

The constant model learns the training encounter prevalence. The age/gender
classifier uses only shared standardised age and the male indicator; female is
the reference. Its coefficients are not regularised or tuned. Structured LR
uses all 27 structured features, including eight missingness indicators, with
L2 regularisation. Structured-plus-text LR adds the 2,772 frozen TF-IDF terms to
the same structured columns, for 2,799 features. Each regularised LR's C is
selected independently from the same six fixed values by maximum validation AP;
ties within 1e-12 favour smaller C. No class weighting or resampling is used.
The text model uses lbfgs, tol 1e-8 and max_iter 20000; the earlier models' settings
are unchanged. No XGBoost-specific sparse-zero correction is applied to LR.


In [ ]:
from copy import deepcopy
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import time
import warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, precision_recall_curve,
    roc_auc_score, roc_curve,
)
from sklearn.pipeline import Pipeline

import aki_preprocessing as ap

TARGET = "aki_label"
MODEL_ORDER = ["prevalence", "age_gender_lr", "structured_lr", "structured_text_lr"]
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
AGE_GENDER_FEATURES = ["num__age", "cat__gender_M"]
CORE_VERSIONS = {
    "numpy": "2.3.5", "pandas": "2.3.3", "scipy": "1.16.3",
    "scikit-learn": "1.7.2", "joblib": "1.5.2",
}


def find_project_dir(project_dir=None):
    if project_dir is not None:
        candidates = [Path(project_dir).expanduser().resolve()]
    else:
        here = Path.cwd().resolve()
        candidates = []
        for base in [here, *here.parents]:
            candidates.extend([base, base / "Group Project"])
    for candidate in candidates:
        if (candidate / "person2_handoff/person2_split_preprocessing/outputs/split_manifest.csv").is_file():
            return candidate
    raise FileNotFoundError("Set project_dir to the folder containing person2_handoff.")


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def check(condition, message, checks):
    checks.append({"check": message, "pass": bool(condition)})
    if not condition:
        raise AssertionError(message)


def development_plan():
    return {
        "date": "2026-10-16", "original_baselines_date": "2026-10-13", "version": 2, "target": TARGET,
        "scope": "Original three primary-outcome baselines plus structured + chief-complaint LR",
        "split": "Person 2 frozen patient split, seed 90089; no new split",
        "preprocessing": "Person 2 fitted lr_structured and lr_text, unchanged",
        "prevalence": "DummyClassifier(strategy='prior'); training encounter prevalence",
        "age_gender": {
            "features": AGE_GENDER_FEATURES, "penalty": None, "tuning": "none",
            "reference": "female (gender in training/validation is restricted to F or M)",
        },
        "structured": {"penalty": "l2", "C_grid": C_GRID},
        "structured_text": {"penalty": "l2", "C_grid": C_GRID,
                            "preprocessor": "lr_text", "features": 2799, "text_terms": 2772,
                            "solver": "lbfgs", "max_iter": 20000, "tol": 1e-8,
                            "numerical_revision": "Initial C=100 fit reached 5,000 iterations without convergence; limit increased to 20,000 for all final text fits; C grid and AP rule unchanged",
                            "search_expansion": "none; fixed before inspecting text-LR tuning results"},
        "optimizer": {"solver": "lbfgs", "max_iter": 2000, "tol": 1e-8},
        "selection": "Maximum validation average precision; ties within 1e-12 choose smaller C",
        "selection_metric_status": "Provisional Person 3 development choice; Person 5 agreement pending",
        "supporting_metrics": ["AUROC", "Brier score", "quantile calibration curve"],
        "weighting": "No class weighting, patient weighting or resampling",
        "threshold": "Pending team agreement; no threshold metrics or threshold selected",
        "calibration": "Assessment only; no probability recalibration fitted",
        "final_fitting": "Training only; no train+validation refit",
        "test": "No test outcomes, predictors, matrices or predictions read/generated",
        "final_team_freeze": "Pending Persons 4 and 5 coordination",
        "secondary_and_sensitivity": "Stage >=2 and sensitivity analyses remain outside this handoff scope",
        "xgboost_correction": "Person 4 responsibility; no XGBoost-specific sparse-zero change applied to LR",
    }

## Handoff validation

Check the manifest, patient separation, identifier order, predictor whitelist,
package versions, saved matrices and training-only preprocessing statistics.
Test patient assignments in the manifest are checked for separation, but no
test outcomes, predictors or matrices are opened. Reproducing the saved matrices
with the original fitted preprocessors also checks prediction row alignment.
Verify identical structured columns between LR variants and identical TF-IDF
settings, vocabulary and IDF between lr_text and tree_text. A fresh training-only
vectorizer fit is used for QA, not as a replacement model input.


In [ ]:
def load_development_data(project_dir):
    checks = []
    for package, version in CORE_VERSIONS.items():
        actual = importlib.metadata.version(package)
        if actual != version:
            raise RuntimeError(f"{package} {actual} != {version}; install requirements.txt and restart the kernel.")
    base = project_dir / "person2_handoff/person2_split_preprocessing"
    outputs = base / "outputs"
    source_module = base / "aki_preprocessing.py"
    check(sha256(source_module) == sha256(ap.__file__), "Preprocessing module is byte-identical to Person 2", checks)
    predictors = pd.read_csv(project_dir / "aki_handoff/predictor_columns.csv")["predictor"].tolist()
    check(predictors == ap.PREDICTORS, "Exactly Person 1/2's 13 permitted predictors", checks)
    manifest_path = outputs / "split_manifest.csv"
    manifest = pd.read_csv(manifest_path)
    check(len(manifest) == 113097, "Manifest covers 113,097 encounters", checks)
    check(manifest.subject_id.nunique() == 66090, "Manifest covers 66,090 patients", checks)
    check(not manifest.stay_id.duplicated().any(), "Unique manifest stay_id", checks)
    check(set(manifest.partition) == {"train", "validation", "test"}, "Only the frozen three partition names", checks)
    check(manifest.groupby("subject_id").partition.nunique().max() == 1, "Zero patient overlap across all partitions", checks)
    pre_path = outputs / "preprocessors/lr_structured.joblib"
    pre = joblib.load(pre_path)
    names_path = outputs / "preprocessors/lr_structured_feature_names.csv"
    names = pd.read_csv(names_path)["feature"].tolist()
    check(list(pre.get_feature_names_out()) == names and len(names) == 27, "Shared structured feature names and 27 columns", checks)
    check(set(pre.feature_names_in_) == set(predictors), "Preprocessor accepts only the permitted predictors", checks)
    check(not any("chiefcomplaint" in n or n.startswith("text__") for n in names), "No chief-complaint text features", checks)
    hashes = {
        str(path.relative_to(project_dir)): sha256(path)
        for path in [manifest_path, pre_path, names_path, source_module,
                     project_dir / "aki_handoff/predictor_columns.csv"]
    }
    data = {"preprocessor": pre, "feature_names": names, "predictors": predictors,
            "source_hashes": hashes, "manifest": manifest}
    text_pre_path = outputs / "preprocessors/lr_text.joblib"
    text_names_path = outputs / "preprocessors/lr_text_feature_names.csv"
    text_pre = joblib.load(text_pre_path)
    text_names = pd.read_csv(text_names_path)["feature"].tolist()
    check(list(text_pre.get_feature_names_out()) == text_names and len(text_names) == 2799,
          "Shared lr_text feature names and 2,799 columns", checks)
    check(text_names[:27] == names, "lr_text preserves the exact 27-column structured prefix", checks)
    check(list(text_pre.feature_names_in_) == predictors, "Text preprocessor accepts exactly the permitted raw inputs", checks)
    data["text_preprocessor"], data["text_feature_names"] = text_pre, text_names
    data["text_matrix_reproduction"] = {}
    for path in (text_pre_path, text_names_path):
        hashes[str(path.relative_to(project_dir))] = sha256(path)
    expected = {"train": (79364, 46263, 5546), "validation": (16932, 9913, 1165)}
    for partition in ("train", "validation"):
        label_path = outputs / f"matrices/labels_{partition}.csv"
        raw_path = outputs / f"matrices/X_raw_{partition}.csv"
        matrix_path = outputs / f"matrices/lr_structured_{partition}.npy"
        # The secondary target is intentionally not loaded.
        labels = pd.read_csv(label_path, usecols=["stay_id", "subject_id", "partition", TARGET])
        raw = pd.read_csv(raw_path)
        matrix = np.load(matrix_path)
        assigned = manifest[manifest.partition.eq(partition)].sort_values("stay_id").reset_index(drop=True)
        check(labels[["subject_id", "stay_id", "partition"]].equals(assigned[["subject_id", "stay_id", "partition"]]),
              f"{partition}: IDs and row order match the frozen manifest", checks)
        check(raw.stay_id.equals(labels.stay_id), f"{partition}: raw predictors align with labels", checks)
        check(list(raw.columns) == ["stay_id"] + predictors, f"{partition}: raw table has exactly the permitted schema", checks)
        check(labels[TARGET].isin([0, 1]).all() and labels[TARGET].notna().all(), f"{partition}: binary primary labels", checks)
        observed = (len(labels), labels.subject_id.nunique(), int(labels[TARGET].sum()))
        check(observed == expected[partition], f"{partition}: encounter, patient and positive counts reconcile", checks)
        check(matrix.shape == (len(labels), 27) and np.isfinite(matrix).all(), f"{partition}: finite matrix with expected shape", checks)
        X = raw[predictors].copy()
        check(set(X.gender) == {"F", "M"}, f"{partition}: gender coding supports the predefined binary baseline", checks)
        reproduced = pre.transform(X)
        check(np.array_equal(reproduced, matrix), f"{partition}: saved preprocessor reproduces every matrix value", checks)
        data[partition] = {"labels": labels, "X": X, "matrix": matrix, "y": labels[TARGET].to_numpy()}
        for path in (label_path, raw_path, matrix_path):
            hashes[str(path.relative_to(project_dir))] = sha256(path)
        text_matrix_path = outputs / f"matrices/lr_text_{partition}.npz"
        text_matrix = sparse.load_npz(text_matrix_path).tocsr()
        check(text_matrix.shape == (len(labels), 2799) and np.isfinite(text_matrix.data).all(),
              f"{partition}: finite sparse lr_text matrix with expected shape", checks)
        text_reproduced = sparse.csr_matrix(text_pre.transform(X))
        difference = text_reproduced - text_matrix
        difference.eliminate_zeros()
        max_error = float(np.max(np.abs(difference.data))) if difference.nnz else 0.0
        check(np.array_equal(text_matrix.indptr, text_reproduced.indptr)
              and np.array_equal(text_matrix.indices, text_reproduced.indices)
              and max_error <= 1e-14,
              f"{partition}: lr_text matrix pattern and values reproduce within 1e-14", checks)
        data["text_matrix_reproduction"][partition] = {"max_abs_difference": max_error, "atol": 1e-14}
        check(np.array_equal(text_matrix[:, :27].toarray(), matrix),
              f"{partition}: text model structured inputs equal structured LR inputs", checks)
        data[partition]["text_matrix"] = text_matrix
        # Fit using the supplied training matrix. For probability output use the
        # fitted preprocessor's local transform, matching the persisted pipeline.
        # Cross-platform sparse normalisation can differ by a few floating ULPs.
        data[partition]["text_prediction_matrix"] = text_reproduced
        hashes[str(text_matrix_path.relative_to(project_dir))] = sha256(text_matrix_path)
    train = data["train"]["X"]
    numeric = pre.named_transformers_["num"]
    imputer = numeric.named_steps["impute"]
    check(np.array_equal(imputer.statistics_, train[ap.NUMERIC].median().to_numpy()), "Imputation medians equal training medians", checks)
    imputed = imputer.transform(train[ap.NUMERIC])
    check(np.allclose(numeric.named_steps["scale"].mean_, imputed.mean(axis=0), atol=1e-12, rtol=0), "Scaler centres equal imputed training means", checks)
    cat = pre.named_transformers_["cat"]
    cat_train = cat.named_steps["impute"].transform(train[ap.CATEGORICAL])
    check(all(np.array_equal(v, np.unique(cat_train[:, i])) for i, v in enumerate(cat.named_steps["onehot"].categories_)),
          "One-hot categories equal training categories", checks)
    vectorizer = text_pre.named_transformers_["text"].named_steps["tfidf"]
    # Independent training-only QA; this refit is not used for model inputs.
    refit = clone(vectorizer).fit(ap.clean_text(train[ap.TEXT]))
    check(vectorizer.vocabulary_ == refit.vocabulary_ and np.array_equal(vectorizer.idf_, refit.idf_),
          "TF-IDF vocabulary and IDF reproduce a training-only refit", checks)
    check(len(vectorizer.vocabulary_) == 2772, "Frozen TF-IDF vocabulary contains 2,772 terms", checks)
    tree_pre_path = outputs / "preprocessors/tree_text.joblib"
    tree_pre = joblib.load(tree_pre_path)
    tree_vec = tree_pre.named_transformers_["text"].named_steps["tfidf"]
    check(vectorizer.get_params() == tree_vec.get_params()
          and vectorizer.vocabulary_ == tree_vec.vocabulary_
          and np.array_equal(vectorizer.idf_, tree_vec.idf_),
          "LR and XGBoost shared text settings, vocabulary and IDF are identical", checks)
    hashes[str(tree_pre_path.relative_to(project_dir))] = sha256(tree_pre_path)
    return data, checks

## Baseline fitting and tuning functions

Models are fitted only on training data. Validation is used to select the
structured and text models' C values and describe development performance. No train/validation
refit, cross-validation, threshold optimisation or probability recalibration is
performed. Convergence warnings stop execution rather than being ignored.


In [ ]:
def probability_metrics(y, probability):
    if len(y) != len(probability) or not np.isfinite(probability).all() or not ((probability >= 0) & (probability <= 1)).all():
        raise ValueError("Invalid prediction vector")
    return {"average_precision": float(average_precision_score(y, probability)),
            "auroc": float(roc_auc_score(y, probability)),
            "brier_score": float(brier_score_loss(y, probability))}


def fit_lr(X, y, penalty="l2", C=1.0, max_iter=2000):
    model = LogisticRegression(penalty=penalty, C=C, solver="lbfgs", max_iter=max_iter,
                               tol=1e-8, class_weight=None, fit_intercept=True)
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)
    if any(issubclass(w.category, ConvergenceWarning) for w in caught):
        raise RuntimeError(f"Logistic regression failed to converge for C={C}, penalty={penalty}")
    return model, time.perf_counter() - start


def develop_baselines(data):
    tr, va = data["train"], data["validation"]
    pre, names = data["preprocessor"], data["feature_names"]
    models, probabilities, tuning = {}, {}, []
    dummy = DummyClassifier(strategy="prior").fit(tr["matrix"], tr["y"])
    models["prevalence"] = Pipeline([("preprocess", deepcopy(pre)), ("model", dummy)])
    probabilities["prevalence"] = dummy.predict_proba(va["matrix"])[:, 1]
    indices = [names.index(feature) for feature in AGE_GENDER_FEATURES]
    selector = ColumnTransformer([("age_gender", "passthrough", indices)], remainder="drop", sparse_threshold=0)
    ag_tr = selector.fit_transform(tr["matrix"])
    ag_va = selector.transform(va["matrix"])
    age_gender, seconds = fit_lr(ag_tr, tr["y"], penalty=None)
    ag_prob = age_gender.predict_proba(ag_va)[:, 1]
    models["age_gender_lr"] = Pipeline([("preprocess", deepcopy(pre)), ("select", selector), ("model", age_gender)])
    probabilities["age_gender_lr"] = ag_prob
    tuning.append({"model_name": "age_gender_lr", "penalty": "none", "C": np.nan,
                   **probability_metrics(va["y"], ag_prob), "n_iter": int(age_gender.n_iter_[0]),
                   "fit_seconds": seconds, "converged": True, "selected": True,
                   "selection_rule": "Fixed two-predictor baseline; no tuning"})
    candidates = []
    for C in C_GRID:
        model, seconds = fit_lr(tr["matrix"], tr["y"], C=C)
        prob = model.predict_proba(va["matrix"])[:, 1]
        row = {"model_name": "structured_lr", "penalty": "l2", "C": C,
               **probability_metrics(va["y"], prob), "n_iter": int(model.n_iter_[0]),
               "fit_seconds": seconds, "converged": True, "selected": False,
               "selection_rule": "Max validation AP; ties within 1e-12 choose smaller C"}
        tuning.append(row)
        candidates.append((model, prob, row))
    best_ap = max(row["average_precision"] for _, _, row in candidates)
    model, prob, selected = min(
        (item for item in candidates if best_ap - item[2]["average_precision"] <= 1e-12),
        key=lambda item: item[2]["C"],
    )
    selected["selected"] = True
    models["structured_lr"] = Pipeline([("preprocess", deepcopy(pre)), ("model", model)])
    probabilities["structured_lr"] = prob
    selected_structured_C = float(selected["C"])
    text_candidates = []
    for C in C_GRID:
        print(f"Fitting structured + text LR, C={C:g}", flush=True)
        model, seconds = fit_lr(tr["text_matrix"], tr["y"], C=C, max_iter=20000)
        prob = model.predict_proba(va["text_prediction_matrix"])[:, 1]
        row = {"model_name": "structured_text_lr", "penalty": "l2", "C": C,
               **probability_metrics(va["y"], prob), "n_iter": int(model.n_iter_[0]),
               "fit_seconds": seconds, "converged": True, "selected": False,
               "selection_rule": "Max validation AP; ties within 1e-12 choose smaller C"}
        tuning.append(row)
        text_candidates.append((model, prob, row))
        print(f"  converged in {row['n_iter']} iterations; validation AP={row['average_precision']:.6f}", flush=True)
    best_text_ap = max(row["average_precision"] for _, _, row in text_candidates)
    text_model, text_prob, selected_text = min(
        (item for item in text_candidates if best_text_ap - item[2]["average_precision"] <= 1e-12),
        key=lambda item: item[2]["C"],
    )
    selected_text["selected"] = True
    models["structured_text_lr"] = Pipeline([("preprocess", deepcopy(data["text_preprocessor"])), ("model", text_model)])
    probabilities["structured_text_lr"] = text_prob
    metrics = pd.DataFrame([
        {"model_name": name, "target": TARGET, "partition": "validation",
         "encounters": len(va["y"]), "patients": va["labels"].subject_id.nunique(),
         "positives": int(va["y"].sum()), "prevalence": float(va["y"].mean()),
         "predictive_features": {"prevalence": 0, "age_gender_lr": 2, "structured_lr": 27, "structured_text_lr": 2799}[name],
         **probability_metrics(va["y"], probabilities[name])}
        for name in MODEL_ORDER
    ])
    return {"models": models, "probabilities": probabilities, "tuning": pd.DataFrame(tuning),
            "metrics": metrics, "selected_C": selected_structured_C,
            "selected_text_C": float(selected_text["C"])}

## Reproducible pipelines and prediction exports

Each saved sklearn Pipeline accepts the original 13 cleaned predictor columns.
The age/gender classifier receives only its two selected columns. The constant
classifier ignores all feature values. Verify that raw-input, matrix-input and
reloaded-pipeline probabilities agree, and that CSV probabilities round-trip.
Predictions retain stay_id and subject_id for Person 5's paired evaluation.


In [ ]:
def save_handoff(project_dir, data, result, checks):
    out = project_dir / "person3_handoff/outputs"
    for folder in (out / "models", out / "figures"):
        folder.mkdir(parents=True, exist_ok=True)
    va = data["validation"]
    prediction_tables, coefficients = [], []
    for name in MODEL_ORDER:
        pipeline = result["models"][name]
        probability = result["probabilities"][name]
        raw_prob = pipeline.predict_proba(va["X"])[:, 1]
        check(np.allclose(raw_prob, probability, atol=1e-12, rtol=0), f"{name}: raw-input pipeline matches matrix predictions", checks)
        path = out / "models" / f"{name}.joblib"
        joblib.dump(pipeline, path, compress=3)
        reloaded = joblib.load(path)
        check(np.array_equal(reloaded.predict_proba(va["X"])[:, 1], raw_prob), f"{name}: reloaded pipeline reproduces validation probabilities exactly", checks)
        pred = va["labels"][["stay_id", "subject_id"]].copy()
        pred["target"] = TARGET
        pred["observed_label"] = va["y"]
        pred["predicted_probability"] = probability
        pred["model_name"] = name
        pred["partition"] = "validation"
        prediction_tables.append(pred)
        if name != "prevalence":
            features = (AGE_GENDER_FEATURES if name == "age_gender_lr"
                        else data["text_feature_names"] if name == "structured_text_lr"
                        else data["feature_names"])
            estimator = pipeline.named_steps["model"]
            for feature, coefficient in zip(features, estimator.coef_[0]):
                coefficients.append({"model_name": name, "feature": feature, "coefficient": coefficient})
            coefficients.append({"model_name": name, "feature": "intercept", "coefficient": estimator.intercept_[0]})
    predictions = pd.concat(prediction_tables, ignore_index=True)
    check(not predictions.duplicated(["stay_id", "model_name", "target"]).any(), "One prediction per validation encounter/model/target", checks)
    check(predictions.groupby("model_name").size().eq(len(va["y"])).all(), "All four baselines evaluate the same validation encounters", checks)
    check(predictions.partition.eq("validation").all() and predictions.target.eq(TARGET).all(), "Only validation predictions for the primary target", checks)
    check(np.all(result["probabilities"]["prevalence"] == data["train"]["y"].mean()), "Constant baseline is exactly training encounter prevalence", checks)
    baseline_ap = float(result["metrics"].query("model_name == 'prevalence'").average_precision.iloc[0])
    check(abs(baseline_ap - va["y"].mean()) < 1e-12, "Constant AP equals validation prevalence", checks)
    predictions.to_csv(out / "validation_predictions.csv", index=False)
    restored = pd.read_csv(out / "validation_predictions.csv", float_precision="round_trip")
    check(np.array_equal(restored.predicted_probability.to_numpy(), predictions.predicted_probability.to_numpy()), "Prediction CSV preserves probabilities on round-trip read", checks)
    result["metrics"].to_csv(out / "validation_metrics.csv", index=False)
    result["tuning"].to_csv(out / "tuning_record.csv", index=False)
    pd.DataFrame(coefficients).to_csv(out / "coefficients.csv", index=False)
    pd.DataFrame(checks).to_csv(out / "handoff_checks.csv", index=False)
    metadata = {
        "development_plan": development_plan(), "selected_structured_C": result["selected_C"],
        "selected_text_C": result["selected_text_C"],
        "text_representation": {"preprocessor": "Person 2 lr_text, unchanged", "text_terms": 2772,
                                "total_features": 2799, "matches_tree_text_vocabulary_and_idf": True},
        "text_matrix_numerical_reproduction": data["text_matrix_reproduction"],
        "train_encounters": len(data["train"]["y"]), "validation_encounters": len(va["y"]),
        "training_prevalence": float(data["train"]["y"].mean()),
        "environment": {"python": platform.python_version(), **{p: importlib.metadata.version(p) for p in CORE_VERSIONS}},
        "source_sha256": data["source_hashes"],
        "test_evaluation": "pending; no test data opened by Person 3 code",
        "prediction_schema_status": "Snake-case names for the seven fields requested by the team methodology; Person 5 can rename if needed",
    }
    (out / "model_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
    return out

## Plotting and report contribution functions

Plot precision–recall, ROC and ten-quantile calibration curves. AP uses
`average_precision_score`, not trapezoidal PR area. Identical predictions in
the constant model form a single calibration point. No confidence bands are
shown; patient-level uncertainty estimation belongs to the shared evaluation.


In [ ]:
def make_validation_figure(out, data, result):
    y = data["validation"]["y"]
    prevalence = float(y.mean())
    colours = {"prevalence": "#707070", "age_gender_lr": "#CB6E32", "structured_lr": "#147C80", "structured_text_lr": "#7355A5"}
    labels = {"prevalence": "Training prevalence", "age_gender_lr": "Age / gender LR", "structured_lr": "Structured LR", "structured_text_lr": "Structured + text LR"}
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), layout="constrained")
    calibration_rows = []
    for name in MODEL_ORDER:
        p = result["probabilities"][name]
        metrics = result["metrics"].set_index("model_name").loc[name]
        if name != "prevalence":
            precision, recall, _ = precision_recall_curve(y, p)
            # AP is the step-weighted score, not trapezoidal area under this display.
            axes[0].step(recall[::-1], precision[::-1], where="pre", color=colours[name],
                         label=f"{labels[name]} · AP {metrics.average_precision:.3f}")
        else:
            axes[0].axhline(prevalence, color=colours[name], ls="--", label=f"Prevalence · AP {prevalence:.3f}")
        fpr, tpr, _ = roc_curve(y, p)
        axes[1].plot(fpr, tpr, color=colours[name], label=f"{labels[name]} · {metrics.auroc:.3f}",
                     ls="--" if name == "prevalence" else "-")
        observed, predicted = calibration_curve(y, p, n_bins=10, strategy="quantile")
        axes[2].plot(predicted, observed, "o-", color=colours[name], label=labels[name])
        for i, (x, z) in enumerate(zip(predicted, observed)):
            calibration_rows.append({"model_name": name, "bin_index": i, "mean_prediction": x, "observed_fraction": z})
    axes[0].set(xlabel="Recall (sensitivity)", ylabel="Precision (PPV)", title="Precision–recall", xlim=(0, 1), ylim=(0, 1))
    axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="ROC", xlim=(0, 1), ylim=(0, 1))
    lim = max(max(row["mean_prediction"], row["observed_fraction"]) for row in calibration_rows) * 1.1
    axes[2].plot([0, lim], [0, lim], ":", color="#909090", label="Ideal")
    axes[2].set(xlabel="Mean predicted probability", ylabel="Observed AKI fraction", title="Calibration · quantile bins", xlim=(0, lim), ylim=(0, lim))
    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(alpha=0.18)
        ax.legend(fontsize=7.5, loc="best", frameon=False)
    fig.suptitle("Primary AKI outcome · validation only · 16,932 encounters / 9,913 patients", fontsize=13)
    fig.savefig(out / "figures/validation_discrimination_calibration.png", dpi=180)
    plt.close(fig)
    pd.DataFrame(calibration_rows).to_csv(out / "calibration_bins.csv", index=False)


def write_report_contribution(project_dir, data, result):
    indexed = result["metrics"].set_index("model_name")
    constant, age, structured, text_lr = (indexed.loc[name] for name in MODEL_ORDER)
    text = f"""# Person 3 baseline methods and validation results

## Methods contribution

Four baseline models were developed for the primary outcome, observed creatinine-defined AKI
within 24 hours of ED arrival or earlier hospital discharge. All models used Person 2's fixed
patient-separated partitions (seed 90089), with 79,364 training encounters from 46,263 patients
and 16,932 validation encounters from 9,913 patients. Person 2's lr_structured and lr_text
preprocessing were reused without change:
training-median numeric imputation, numeric standardisation, missingness indicators, and
one-hot categorical encoding. Identifiers, laboratory measurements and ascertainment fields
were excluded from model inputs. The fitted preprocessing was verified against the saved
training and validation matrices.

The prevalence baseline assigned every encounter the training outcome frequency
({data['train']['y'].mean():.4%}). An unpenalised logistic regression used standardised age and
the male-gender indicator only, with female as reference. Structured logistic regression used
all 27 transformed structured features and an L2 penalty. Six regularisation values
(C = 0.001, 0.01, 0.1, 1, 10, 100) were compared, selecting maximum validation average
precision (AP); ties within 1e-12 favoured stronger regularisation. The selected C was
{result['selected_C']:g} for structured LR. The additional structured-plus-text model used
the same 27 structured columns plus 2,772 frozen training TF-IDF terms, for 2,799 columns.
Text was lowercased and represented by 1-2 grams with min_df 5 and sublinear term frequency,
using Person 2's tokenizer. The vocabulary, IDF weights and settings were verified to match
Person 2's tree_text representation. Its C was selected independently from the same six
values using the same AP rule; the selected value was {result['selected_text_C']:g}. Both
LR models used lbfgs with tol 1e-8; iteration limits were 2,000 for structured LR and 20,000
for text LR. The text limit was increased from 5,000 after the first C = 100 fit
did not converge; no candidate values, tolerance or selection rules were changed.
Models were fitted on training data only, without resampling or
class weighting. AP, AUROC, Brier score and calibration curves were calculated on validation
data. AP is the recall-increment-weighted metric implemented by average_precision_score,
not a trapezoidal precision–recall area. All logistic regressions converged.

## Validation results contribution

The validation set contained 1,165 positive encounters (6.8805%). The prevalence baseline
achieved AP {constant.average_precision:.4f}, AUROC {constant.auroc:.4f}, and Brier score
{constant.brier_score:.4f}. Age/gender logistic regression achieved AP {age.average_precision:.4f},
AUROC {age.auroc:.4f}, and Brier score {age.brier_score:.4f}. Structured logistic regression
achieved AP {structured.average_precision:.4f}, AUROC {structured.auroc:.4f}, and Brier score
{structured.brier_score:.4f}. Structured-plus-text LR achieved AP {text_lr.average_precision:.4f},
AUROC {text_lr.auroc:.4f}, and Brier score {text_lr.brier_score:.4f}. The within-LR text
comparison changed AP by {text_lr.average_precision - structured.average_precision:+.4f}
and AUROC by {text_lr.auroc - structured.auroc:+.4f}, with a Brier-score change of
{text_lr.brier_score - structured.brier_score:+.5f}. See outputs/validation_metrics.csv
and the validation figure for complete values. These are encounter-level development
results: both regularised LR models were selected using this validation set, so their
performance is not an independent
final estimate. Repeated encounters also require patient-level resampling for uncertainty
estimates; confidence intervals have not been calculated in this handoff.

## Interpretation and work remaining

Age/gender alone added little discrimination (AUROC {age.auroc:.3f}). Structured predictors
increased AP to {structured.average_precision / constant.average_precision:.2f} times the
prevalence baseline, but AUROC {structured.auroc:.3f} still indicates modest discrimination.
The Brier reduction was only {constant.brier_score - structured.brier_score:.5f} in absolute
terms ({100 * (constant.brier_score - structured.brier_score) / constant.brier_score:.2f}%
relative). The text comparison keeps the structured columns, outcome, encounters and
partition fixed, while allowing each model's penalty to be selected by the same predefined
search. It assesses adding text within the LR modelling approach. A common-C comparison
is also available in outputs/tuning_record.csv. It does not establish clinical utility
or a statistically reliable improvement. In the original structured search, AP for
C = 10 and 100 differed by less than 0.000001; selection followed the recorded rule
and should not be interpreted as evidence for a distinct optimal penalty. Calibration
plots summarise quantile groups and lack uncertainty intervals; useful probability
calibration and clinical operating thresholds still require shared evaluation.

This study identifies observed AKI,
including cases already positive at the first sample, in a selected outcome-assessable
admitted population. Operational negatives do not establish clinical absence of AKI.
Regression coefficients describe associations and depend on coding and regularisation.
Chief complaints may contain renal diagnoses, abnormal laboratory findings or clinical
suspicions already known at presentation. Their predictive association does not show that
the model anticipates AKI before any clinical suspicion. Early identification is the
appropriate task description.

Comparison with text-inclusive XGBoost must use Person 4's corrected input representation
and updated results; the older dense-versus-sparse comparison also changed structured-zero
handling. No such correction is needed for LR. Algorithm-specific numeric preprocessing
can differ, while encounters, targets and the frozen text representation remain aligned.

Person 5 should agree the final metric and threshold protocol with Persons 3 and 4 before
final test evaluation. AP is currently a provisional Person 3 tuning choice. No threshold
has been selected, no probability recalibration has been fitted, and no test predictors,
labels, matrices or predictions were opened or generated by this code. Stage >=2 and
sensitivity evaluations remain outside the present scope. The text-LR addition was
authorised on 16 October 2026 following the team discussion; the earlier three-model
handoff is retained separately, and baseline preservation checks compare its original
prediction and metric fingerprints with the updated outputs.

## Implementation references

- [scikit-learn 1.7.2 LogisticRegression](https://scikit-learn.org/1.7/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [scikit-learn 1.7.2 average_precision_score](https://scikit-learn.org/1.7/modules/generated/sklearn.metrics.average_precision_score.html)
- [scikit-learn 1.7.2 DummyClassifier](https://scikit-learn.org/1.7/modules/generated/sklearn.dummy.DummyClassifier.html)

Proposed CRediT contribution: Methodology, Software, Formal analysis, Validation,
Visualisation, Writing – original draft (baseline methods and results).
"""
    (project_dir / "person3_handoff/P3_report_contribution.md").write_text(text, encoding="utf-8")

## Complete workflow helper

`run(PROJECT_DIR)` is the command-line equivalent of the execution cells below.
Do not call it again after those cells unless deliberately rerunning the workflow.


In [ ]:
def run(project_dir=None):
    project_dir = find_project_dir(project_dir)
    out = project_dir / "person3_handoff/outputs"
    out.mkdir(parents=True, exist_ok=True)
    # Persist this bounded plan BEFORE looking at any tuning performance.
    (out / "analysis_plan.json").write_text(json.dumps(development_plan(), indent=2) + "\n")
    data, checks = load_development_data(project_dir)
    result = develop_baselines(data)
    out = save_handoff(project_dir, data, result, checks)
    make_validation_figure(out, data, result)
    write_report_contribution(project_dir, data, result)
    print(result["metrics"].to_string(index=False))
    print(f"Selected structured C: {result['selected_C']:g}; checks passed: {len(checks)} / {len(checks)}")
    print("Test evaluation: pending; development code only opens train and validation data.")
    return data, result, pd.DataFrame(checks), out

## Execute checks and save the plan before tuning

Only aggregate results are displayed. The runtime source hashes identify the
exact upstream artifacts used in this run.


In [ ]:
from IPython.display import display, Image, Markdown

project_root = find_project_dir(PROJECT_DIR)
out = project_root / "person3_handoff/outputs"
out.mkdir(parents=True, exist_ok=True)
(out / "analysis_plan.json").write_text(json.dumps(development_plan(), indent=2) + "\n")
data, checks = load_development_data(project_root)
display(pd.DataFrame(checks))
display(pd.DataFrame([
    {"partition": part, "encounters": len(data[part]["y"]),
     "patients": data[part]["labels"].subject_id.nunique(),
     "positives": int(data[part]["y"].sum()),
     "prevalence": data[part]["y"].mean()}
    for part in ["train", "validation"]
]))


## Fit models and inspect the complete bounded tuning record

In [ ]:
result = develop_baselines(data)
display(result["tuning"])
print("Selected structured C:", result["selected_C"])
print("Selected structured + text C:", result["selected_text_C"])
display(result["metrics"])


## Save the private handoff and verify pipeline/CSV reloads

In [ ]:
out = save_handoff(project_root, data, result, checks)
make_validation_figure(out, data, result)
write_report_contribution(project_root, data, result)
display(pd.DataFrame(checks))
print(f"All {len(checks)} checks passed.")
print("No test outcomes, predictors, matrices or predictions opened/generated.")


## Validation discrimination and calibration

In [ ]:
display(Image(filename=str(out / "figures/validation_discrimination_calibration.png")))

## Methods, validation results and interpretation for report integration

In [ ]:
display(Markdown((project_root / "person3_handoff/P3_report_contribution.md").read_text()))

## Handoff status and next steps

The four training-fitted pipelines and validation probabilities are ready for
review. Person 3's development choices are recorded; the team-wide model freeze
and final test evaluation are pending. No test evaluation is performed here.

Person 5 should reconcile the seven prediction field names, agree AP and a
validation-based threshold rule with Persons 3/4, and use patient-level bootstrap
resampling for uncertainty and paired model comparisons. The 90% sensitivity rule
in `DECISIONS_P3.md` is a proposal only. No threshold has been selected.

The selected model's validation performance is a development estimate because
this set was used to choose C. Higher AP does not by itself establish clinical
utility. Observed AKI includes first-sample positives, and operational negatives
may miss unobserved disease. Keep row-level predictions and fitted models in
approved private storage, consistent with the team handoff.

Chief-complaint terms may reflect conditions already suspected at presentation;
they do not demonstrate prediction before clinical suspicion. Compare against
Person 4's corrected XGBoost representation and updated validation results.
Both algorithms retain the same frozen text vocabulary and TF-IDF settings.

Files: `outputs/validation_metrics.csv`, `outputs/tuning_record.csv`,
`outputs/validation_predictions.csv`, `outputs/models/*.joblib`,
`outputs/handoff_checks.csv`, `outputs/model_metadata.json`,
`P3_report_contribution.md`, `DECISIONS_P3.md` and `README_P3.md`.
